# Lesson 1.2 - Data Structures & Algorithms in Python

## Objectives
- Compare data structure choices using complexity and runtime evidence.
- Implement search, sorting, BFS, and DFS.
- Connect complexity decisions to business latency/cost outcomes.

In [1]:
from collections import deque
import random
import statistics
import time

random.seed(42)

## Section A - Core Data Structures

We compare practical usage of list, tuple, set, dict, deque.

In [2]:
numbers = list(range(8))
immutable_pair = ("user_12", 0.83)
seen = set(["evt_1", "evt_2", "evt_2"])
counts = {"click": 10, "purchase": 2}

stack = []
queue = deque()

for x in [1, 2, 3]:
    stack.append(x)
    queue.append(x)

print("list slice:", numbers[2:6])
print("tuple:", immutable_pair)
print("set:", seen)
print("dict:", counts)
print("stack LIFO:", [stack.pop() for _ in range(3)])
print("queue FIFO:", [queue.popleft() for _ in range(3)])

list slice: [2, 3, 4, 5]
tuple: ('user_12', 0.83)
set: {'evt_1', 'evt_2'}
dict: {'click': 10, 'purchase': 2}
stack LIFO: [3, 2, 1]
queue FIFO: [1, 2, 3]


## Section B - Searching Algorithms

- Linear search: O(n)
- Binary search: O(log n) on sorted arrays

In [3]:
def linear_search(arr: list[int], target: int) -> int:
    for idx, value in enumerate(arr):
        if value == target:
            return idx
    return -1


def binary_search(sorted_arr: list[int], target: int) -> int:
    left, right = 0, len(sorted_arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if sorted_arr[mid] == target:
            return mid
        if sorted_arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1


arr = list(range(200_000))
target = 199_999

t0 = time.perf_counter()
idx_linear = linear_search(arr, target)
t_linear = time.perf_counter() - t0

t0 = time.perf_counter()
idx_binary = binary_search(arr, target)
t_binary = time.perf_counter() - t0

print("linear idx/time:", idx_linear, round(t_linear, 6))
print("binary idx/time:", idx_binary, round(t_binary, 6))
print("speedup ratio:", round(t_linear / max(t_binary, 1e-12), 2))

linear idx/time: 199999 0.00422
binary idx/time: 199999 2.8e-05
speedup ratio: 149.83


## Section C - Sorting: Insertion vs Built-in Timsort

In [4]:
def insertion_sort(values: list[int]) -> list[int]:
    arr = values[:]
    for i in range(1, len(arr)):
        key = arr[i]
        j = i - 1
        while j >= 0 and arr[j] > key:
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
    return arr


data = [random.randint(0, 50_000) for _ in range(3_000)]

t0 = time.perf_counter()
s1 = insertion_sort(data)
t_ins = time.perf_counter() - t0

t0 = time.perf_counter()
s2 = sorted(data)
t_builtin = time.perf_counter() - t0

print("equal results:", s1 == s2)
print("insertion_sort sec:", round(t_ins, 4))
print("sorted sec:", round(t_builtin, 4))

equal results: True
insertion_sort sec: 0.1016
sorted sec: 0.0003


## Section D - Graph Traversal (BFS / DFS)

In [5]:
graph = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F"],
    "D": [],
    "E": ["F"],
    "F": [],
}


def dfs_iterative(g: dict[str, list[str]], start: str) -> list[str]:
    visited = set()
    order = []
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for nb in reversed(g[node]):
            if nb not in visited:
                stack.append(nb)
    return order


def bfs(g: dict[str, list[str]], start: str) -> list[str]:
    visited = {start}
    q = deque([start])
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for nb in g[node]:
            if nb not in visited:
                visited.add(nb)
                q.append(nb)
    return order


print("DFS:", dfs_iterative(graph, "A"))
print("BFS:", bfs(graph, "A"))

DFS: ['A', 'B', 'D', 'E', 'F', 'C']
BFS: ['A', 'B', 'C', 'D', 'E', 'F']


## Section E - Complexity Measurement Experiment

We benchmark linear vs binary search for increasing n.

In [6]:
def benchmark_search(n: int, repeats: int = 8) -> dict[str, float]:
    arr = list(range(n))
    target = n - 1
    linear_times = []
    binary_times = []

    for _ in range(repeats):
        s = time.perf_counter()
        linear_search(arr, target)
        linear_times.append(time.perf_counter() - s)

        s = time.perf_counter()
        binary_search(arr, target)
        binary_times.append(time.perf_counter() - s)

    return {
        "n": float(n),
        "linear_ms": statistics.mean(linear_times) * 1000,
        "binary_ms": statistics.mean(binary_times) * 1000,
    }


rows = [benchmark_search(n) for n in [10_000, 100_000, 500_000, 1_000_000]]
for row in rows:
    print({k: round(v, 4) for k, v in row.items()})

{'n': 10000.0, 'linear_ms': 0.2101, 'binary_ms': 0.0013}
{'n': 100000.0, 'linear_ms': 2.1793, 'binary_ms': 0.0028}
{'n': 500000.0, 'linear_ms': 10.908, 'binary_ms': 0.0037}
{'n': 1000000.0, 'linear_ms': 22.1095, 'binary_ms': 0.0052}


## Business Case Studies & Exceptions

- Log aggregation: replacing nested loops with dict/set patterns can move runtimes from near O(n^2) to near O(n).
- Routing queues: use deque for FIFO to avoid list-shift overhead.
- Exception: for very small bounded datasets, a simpler slower algorithm can be acceptable if readability is materially better.

## Interview Questions & Answers

1. Why is binary search faster than linear search on sorted data?
   It halves the search space each step.
2. When should you use BFS over DFS?
   When shortest unweighted path is required.
3. Why track both time and space complexity?
   A faster algorithm can still fail due to memory overhead.